# HW7 — Finding a signal you have not been told about

Full instructions are in this assignment's `README.md`. Read it first.

You will use a large language model throughout this assignment, and you are expected to.
**What is being graded is not the code — it is how you checked it.**

Keep a record of every prompt as you go; there is a section at the bottom for the log, but
paste them in as you work rather than trying to reconstruct them at the end.


---
# Part 0 — Find the bug yourself, then ask

Below is a function that finds the dominant frequency in a signal. **It runs without error.
It is wrong.**

Do the steps in order. The order is the whole point — once you have seen the answer you
cannot un-see it, and the useful information is in what you predicted beforehand.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def find_dominant_frequency(signal, dt):
    """Return the frequency of the strongest component in a real signal.

    signal : 1-D array of samples
    dt     : time between samples, in seconds
    """
    spectrum = np.abs(np.fft.rfft(signal))
    freqs = np.linspace(0, 1 / dt, len(spectrum))
    return freqs[np.argmax(spectrum[1:]) + 1]


### 0.1 Predict first

Here is a test signal at a frequency you know. **Before running anything below**, write your
predicted output in the cell underneath — an actual number, not "about right".

```python
fs = 1000.0                      # sampling rate, Hz
t  = np.arange(0, 4.0, 1/fs)
x  = np.sin(2*np.pi*50.0*t)      # a 50 Hz sinusoid
```

**Your prediction for `find_dominant_frequency(x, 1/fs)`:**

*(write it here, then move on)*


In [ ]:
fs = 1000.0
t = np.arange(0, 4.0, 1 / fs)
rng = np.random.default_rng(0)
x = np.sin(2 * np.pi * 50.0 * t) + 0.3 * rng.normal(size=t.size)

print(f'reported: {find_dominant_frequency(x, 1/fs):.2f} Hz')


### 0.2 Was your prediction right?

If not — good. Write down what you expected and what you got, before you know why.


### 0.3 Find it by hand

**No LLM for this step.** Work out what the function is actually computing.

A suggestion: plot the spectrum against the frequency axis the function builds, and mark
where you *know* the peak should be. Where does the axis stop being right?


### 0.4 Fix it

Write a corrected version and show that it recovers 50 Hz. Check it on at least two other
frequencies too — a fix that works for one number might be a coincidence.


### 0.5 Now ask a model

Give the **original buggy function** to an LLM and ask it to find the problem. Paste the
prompt and the reply below.

Then answer:

- Did it find the same bug you did?
- Did it find a *different* bug — and if so, is that bug real?
- Did it invent a problem that was not there?
- Was its **explanation** correct, even where its fix was? (These come apart more often than
  you would think.)


### 0.6 Reflection — this is graded

**What did you predict in 0.1, and what was actually returned?**

**If you were wrong, why do you think you missed it?** Be specific and be honest. "The plot
looked reasonable so I did not check the axis" is a better answer than a tidy one — it names
a habit that will cost you again.

*(answer here)*


---
# Part 1 — Learn something niche, with help

Your data is **dispersed**. Radio waves from a distant source travel through the ionised
interstellar medium, and lower frequencies arrive later. Across your observing band the delay
is large enough to smear the signal out completely.

Use an LLM to teach yourself about dispersion and dedispersion, then answer below **in your
own words**.


### 1.1 What causes the delay, and how does it scale with frequency?
*(answer here)*


### 1.2 The delay equation

Write it down. Define every symbol. Say where the constant comes from and **what units it
is in** — this is the single most common place to go wrong.

*(answer here)*


### 1.3 Verify a claim independently

Pick at least one thing the model told you and check it against something that is **not the
model** — a textbook, a review paper, lecture notes.

- What did you check?
- What source did you use?
- Did it agree? If not, who was right?

*(answer here)*


---
# Part 2 — Build the analysis

Use the LLM to help you write the code. Paste your prompts into the log as you go.

**A rule for this part:** do not ask the model to verify its own code. If it writes the
dedispersion, it does not get to be the thing that tells you the dedispersion is right.


### 2.1 Read the file

`data/search_data_01.fil` is **SIGPROC filterbank** format: a header made of length-prefixed
key/value pairs, followed by raw samples.

> **Sanity check:** a correct parser reports **96 frequency channels** and a sampling time of
> **72 µs**. If you get something else, your parser is wrong — do not proceed until it agrees.

Look at what else the header tells you. Some of it you will need for Part 2.2, and at least
one field decides whether your samples are being read correctly at all.


In [ ]:
DATA = '../data/search_data_01.fil'



### 2.2 Dedisperse

For a trial dispersion measure, shift each frequency channel by the appropriate delay and sum
across channels to get a single time series.

Two things worth thinking about before you write it:

- Which direction do the shifts go? Check the sign of the channel spacing in the header
  before you assume.
- Over what range of trial DMs should you search, and how finely? Guessing is fine to start,
  but say why you chose what you chose.


### 2.3 Search for periodicity

For each dedispersed time series, look for a periodic signal. You have done this before in
the course — that code is fair game and is probably better suited to this data than something
written fresh.

What does the periodogram look like at the wrong DM, compared with the right one? That
contrast is your evidence.


### 2.4 Fold

Fold the data at your best candidate period and plot the average profile. A real signal
produces a stable pulse; noise does not.

Fold at a slightly wrong period too, and show what happens. That comparison is worth more
than the correct plot on its own.


---
# Part 3 — Report what you found

State your best estimates, **with uncertainties**, and show the evidence behind them.


In [ ]:
# Report your results here, e.g.
# print(f'DM     = {dm:.2f} +/- {dm_err:.2f} pc/cm^3')
# print(f'period = {period*1e3:.5f} +/- {period_err*1e3:.5f} ms')


### 3.1 How did you get those uncertainties?

Not "it looked about right". What sets the precision of each measurement? For the period, the
length of the observation matters; for the DM, the spacing of your search grid and the width
of the peak.

*(answer here)*


---
# Part 4 — Check it against physics the analysis never saw

The source is real and catalogued. **Your two numbers are enough to identify it.**

Search a pulsar catalogue — the [ATNF Pulsar Catalogue](https://www.atnf.csiro.au/research/pulsar/psrcat/)
is the standard — for an object matching your period and dispersion measure.


### 4.1 Did you find it? What is it?
*(answer here)*


### 4.2 How well do your measurements agree with the published values?

Give the numbers side by side, and the difference.

*(answer here)*


### 4.3 Is any disagreement larger than your uncertainty?

If your measurement and the catalogue disagree by more than you can explain with measurement
error, then something real is unmodelled. **What?**

This has an actual answer. Finding it is worth more than getting the period right was.

*(answer here)*


---
# Graduate Students

### G.1 The telescope was moving
Your period was measured from an observatory on a rotating, orbiting Earth. Estimate the size
of that effect for this observation, correct for it, and say whether the agreement improves.


### G.2 How fine does the DM grid need to be?
Derive the spacing at which two trial DMs would smear the pulse by about one sample — from
this observation's own parameters, not from a lookup — then compare with what you actually
used.


### G.3 Harmonics
Real pulse profiles are not sinusoidal, so power is spread across harmonics of the
fundamental and a plain periodicity search under-counts it. Describe what you would do about
that. If you can, do it, and say whether the detection improves.


---
# Prompt log

Every prompt you sent, in order, with the model named. **Do not tidy them up.** The dead ends
and the re-phrasings are the part that shows how you worked.

```
1. [model]  prompt...
   -> what it gave you, and what you did with it

2. [model]  prompt...
   ->
```

*(log here)*
